In [1]:
from dataPrep import DatasetPreparation
from model import ConvModel
from gradcam import gradCam
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [4]:
vgg_backbone = tf.keras.applications.vgg16.VGG16(
    include_top= False,
    weights= 'imagenet',
    input_shape=(256,256,3) 
)

In [5]:
vgg_backbone.summary()

Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 256, 256, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 256, 256, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 128, 128, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 128, 128, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 128, 128, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 64, 64, 128)       0     

In [6]:
feature_map = [layer.output for layer in vgg_backbone.layers[1:]]
feature_map_model = tf.keras.Model(
    inputs = vgg_backbone.input,
    outputs = feature_map
)
feature_map_model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 256, 256, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 256, 256, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 128, 128, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 128, 128, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 128, 128, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 64, 64, 128)       0     

In [7]:

train_path = r"D:\RECURSOS DE TRABAJO\Base de Datos para IA\Emotions Dataset\Emotions Dataset\train"

CONFIGURATION = {
    "IM_SIZE":256,
    "CLASS_NAMES": ["angry","happy","sad"],
    "BATCH_SIZE":32,
    "SEED":123,
}

prep = DatasetPreparation()
train, val, test = prep.load_all(train_path, CONFIGURATION)

Found 6799 files belonging to 3 classes.
Using 5440 files for training.
Found 6799 files belonging to 3 classes.
Using 1359 files for validation.
Found 2278 files belonging to 3 classes.


In [8]:
# # Ruta completa de la imagen
# ruta_imagen = r'D:\RECURSOS DE TRABAJO\Base de Datos para IA\Emotions Dataset\Emotions Dataset\train\angry\8853.jpg_brightness_2.jpg'

# test_image = cv2.imread(ruta_imagen)

# if test_image is None:
#     print("No se encontró la imagen. Verifica la ruta y el nombre del archivo.")
# else:
#     SIZE = (224, 224)
#     test_image = cv2.resize(test_image, SIZE)
#     im = tf.convert_to_tensor(test_image, dtype=tf.float32)
#     im = im / 255.0
#     img_array = tf.expand_dims(im, axis=0)
#     print(img_array.shape)

#     # Cargar el modelo (ajusta la ruta a tu modelo)
#     model = tf.keras.models.load_model('C:/Users/vpn/Documents/GitHub/Ciencia-de-datos/Deteccion_de_Expresiones_Humanas/modelo.h5')

#     # Método Grad-CAM
#     def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
#         grad_model = tf.keras.models.Model(
#             [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
#         )
#         with tf.GradientTape() as tape:
#             conv_outputs, predictions = grad_model(img_array)
#             if pred_index is None:
#                 pred_index = tf.argmax(predictions[0])
#             class_channel = predictions[:, pred_index]
#         grads = tape.gradient(class_channel, conv_outputs)
#         pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
#         conv_outputs = conv_outputs[0]
#         heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
#         heatmap = tf.squeeze(heatmap)
#         heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
#         return heatmap.numpy()

#     # Cambia 'last_conv_layer_name' por el nombre de la última capa convolucional de tu modelo
#     last_conv_layer_name = 'conv2d'  # Ajusta según tu modelo

#     heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)

#     # Visualización
#     plt.matshow(heatmap)
#     plt.title("Grad-CAM Heatmap")
#     plt.show()

In [9]:
#Obtener la ultima capa del modelo convulsional
def obtener_ultima_capa(model):
    for layer in reversed(model.layers):
        if isinstance(layer,tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No se encontro la capa")

In [11]:
#Construir modelo auxiliar para el GradCam se necesita salida de la ultima convulsion y la ultima del modelo en si

def creando_gradCam_model(model,nombre_ultima_capa):
    ultima_capa_convulsional = model.get_layer(nombre_ultima_capa)

    grad_model = tf.keras.models.Model(
        input = model.input,
        outputs = [ultima_capa_convulsional.output, model.output]
    )

    return grad_model

In [ ]:
#Calculo de gradientes
def crear_heatmap(model, img_array, last_conv_layer_name, class_index=None):

    grad_model = creando_gradCam_model(model=model, nombre_ultima_capa=last_conv_layer_name)

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(img_array)

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        class_chanel = predictions[:, class_index]

    #Gradientes
    grads = tape.gradient(class_chanel,conv_outputs)

    #Promedio espacial
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    #Multiplicar pesos por mapas
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # ReLU
    heatmap = tf.maximum(heatmap, 0)

    # Normalizar
    heatmap /= tf.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
#Superponer el heatmap

def superponer(img_path,heathmap,alpha=0.04):

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = heatmap * alpha + img
    superimposed_img = np.uint8(superimposed_img)

    plt.imshow(superimposed_img)
    plt.axis("off")
    plt.show()


In [13]:
def preprocess_image(img_path, target_size):

    img = tf.keras.preprocessing.image.load_img(
        img_path, target_size=target_size
    )
    
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    img_array = tf.keras.applications.vgg16.preprocess_input(img_array)

    return img_array

In [ ]:
model = tf.keras.applications.vgg16(weights="imagenet")

last_conv_layer_name = obtener_ultima_capa(model)

img_path = "test.jpg"
img_array = preprocess_image(img_path, (224, 224))

heatmap = crear_heatmap(
    model,
    img_array,
    last_conv_layer_name
)

superponer(img_path, heatmap)